# 🐾 Preprocesamiento — Austin Animal Center
**Target:** `outcome_type`

### Correcciones aplicadas vs versión anterior:
- ✅ `sex` y `repro_status` → desde `sex_upon_intake` (no outcome)
- ✅ `age_days` → desde `age_upon_intake` (no outcome)
- ✅ `is_mix` → prioriza `breed_intake` (no breed_outcome)
- ✅ `season` → desde mes de **ingreso** (no outcome)
- ✅ `is_weekend` → desde `datetime_intake` (no outcome)

**Pasos:**
1. Carga de intakes y outcomes
2. Merge de datasets
3. Eliminación de duplicados
4. Tratamiento de nulos
5. Feature engineering
6. Exportar dataset limpio

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('✅ Librerías cargadas')

✅ Librerías cargadas


---
## 1. Carga de datasets

In [9]:
INTAKES_PATH  = '../data/raw/Austin_Animal_Center_Intakes.csv'
OUTCOMES_PATH = '../data/raw/Austin_Animal_Center_Outcomes.csv'

intakes  = pd.read_csv(INTAKES_PATH,  low_memory=False)
outcomes = pd.read_csv(OUTCOMES_PATH, low_memory=False)

print(f'Intakes  → {intakes.shape[0]:,} filas × {intakes.shape[1]} columnas')
print(f'Outcomes → {outcomes.shape[0]:,} filas × {outcomes.shape[1]} columnas')
print('\n=== COLUMNAS INTAKES ===')
print(intakes.columns.tolist())
print('\n=== COLUMNAS OUTCOMES ===')
print(outcomes.columns.tolist())

Intakes  → 124,120 filas × 12 columnas
Outcomes → 124,491 filas × 12 columnas

=== COLUMNAS INTAKES ===
['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Found Location', 'Intake Type', 'Intake Condition', 'Animal Type', 'Sex upon Intake', 'Age upon Intake', 'Breed', 'Color']

=== COLUMNAS OUTCOMES ===
['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Date of Birth', 'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome', 'Age upon Outcome', 'Breed', 'Color']


---
## 2. Normalización de nombres de columnas

In [10]:
def clean_col_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[\s\(\)/]+', '_', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )
    return df

intakes  = clean_col_names(intakes)
outcomes = clean_col_names(outcomes)

print('Columnas intakes  :', intakes.columns.tolist())
print('Columnas outcomes :', outcomes.columns.tolist())

Columnas intakes  : ['animal_id', 'name', 'datetime', 'monthyear', 'found_location', 'intake_type', 'intake_condition', 'animal_type', 'sex_upon_intake', 'age_upon_intake', 'breed', 'color']
Columnas outcomes : ['animal_id', 'name', 'datetime', 'monthyear', 'date_of_birth', 'outcome_type', 'outcome_subtype', 'animal_type', 'sex_upon_outcome', 'age_upon_outcome', 'breed', 'color']


---
## 3. Merge: Intakes + Outcomes

In [11]:
shared_non_key = [c for c in intakes.columns
                  if c in outcomes.columns and c != 'animal_id']
print('Columnas compartidas (sufijo _intake/_outcome):', shared_non_key)

intakes  = intakes.rename(columns={c: f'{c}_intake'  for c in shared_non_key})
outcomes = outcomes.rename(columns={c: f'{c}_outcome' for c in shared_non_key})

df = pd.merge(outcomes, intakes, on='animal_id', how='inner')

print(f'Dataset combinado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

Columnas compartidas (sufijo _intake/_outcome): ['name', 'datetime', 'monthyear', 'animal_type', 'breed', 'color']
Dataset combinado: 160,428 filas × 23 columnas


,animal_id,name_outcome,datetime_outcome,monthyear_outcome,date_of_birth,outcome_type,outcome_subtype,animal_type_outcome,sex_upon_outcome,age_upon_outcome,...,datetime_intake,monthyear_intake,found_location,intake_type,intake_condition,animal_type_intake,sex_upon_intake,age_upon_intake,breed_intake,color_intake
0,A794011,Chunk,05/08/2019 06:20:00 PM,05/08/2019 06:20:00 PM,05/02/2017,Rto-Adopt,NaN,Cat,Neutered Male,2 years,...,05/02/2019 04:51:00 PM,05/02/2019 04:51:00 PM,Austin (TX),Owner Surrender,Normal,Cat,Neutered Male,2 years,Domestic Shorthair Mix,Brown Tabby/White
1,A776359,Gizmo,07/18/2018 04:02:00 PM,07/18/2018 04:02:00 PM,07/12/2017,Adoption,NaN,Dog,Neutered Male,1 year,...,07/12/2018 12:46:00 PM,07/12/2018 12:46:00 PM,7201 Levander Loop in Austin (TX),Stray,Normal,Dog,Intact Male,1 year,Chihuahua Shorthair Mix,White/Brown
2,A821648,NaN,08/16/2020 11:38:00 AM,08/16/2020 11:38:00 AM,08/16/2019,Euthanasia,NaN,Other,Unknown,1 year,...,08/16/2020 10:10:00 AM,08/16/2020 10:10:00 AM,Armadillo Rd And Clubway Ln in Austin (TX),Wildlife,Sick,Other,Unknown,1 year,Raccoon,Gray


---
## 4. Eliminación de duplicados

In [12]:
n_before = len(df)
df = df.drop_duplicates()
print(f'Filas antes : {n_before:,}')
print(f'Filas después: {len(df):,}')
print(f'Duplicados eliminados: {n_before - len(df):,}')

Filas antes : 160,428
Filas después: 160,358
Duplicados eliminados: 70


#### ── Eliminar duplicados tras el merge ────
Un animal puede tener múltiples intakes → el merge genera duplicados
Estrategia: quedarse con el intake MÁS RECIENTE por cada outcome

In [13]:
df['datetime_intake'] = pd.to_datetime(df['datetime_intake'], errors='coerce')
df['datetime_outcome'] = pd.to_datetime(df['datetime_outcome'], errors='coerce')

before = len(df)

# Ordenar por animal_id y fecha de intake descendente
df = df.sort_values(['animal_id', 'datetime_intake'], ascending=[True, False])

# Quedarse con el intake más reciente para cada outcome
df = df.drop_duplicates(subset=['animal_id', 'datetime_outcome'], keep='first')

after = len(df)
print(f'Filas antes : {before:,}')
print(f'Filas después: {after:,}')
print(f'Duplicados eliminados: {before - after:,}')

Filas antes : 160,358
Filas después: 123,660
Duplicados eliminados: 36,698


In [14]:
# ── Diagnóstico de duplicados con fechas ──────────────────────────────────────
# Columnas clave para identificar si son animales distintos o reincidentes
cols_check = ['animal_id', 'datetime_intake', 'datetime_outcome', 
            'animal_type_intake', 'breed_intake', 'outcome_type']
cols_check = [c for c in cols_check if c in df.columns]

# Encontrar grupos con mismo perfil (sin animal_id)
perfil_cols = ['animal_type_intake', 'breed_intake', 'color_intake',
            'sex_upon_intake', 'intake_type', 'intake_condition', 'age_days']

df['perfil_duplicado'] = df.duplicated(subset=perfil_cols, keep=False)

dups = df[df['perfil_duplicado'] == True].sort_values(['breed_intake', 'datetime_intake'])

print(f'Filas con perfil idéntico: {len(dups):,}')
print(f'Tienen el mismo animal_id: {dups.duplicated(subset=["animal_id","datetime_intake"]).sum():,}')
print(f'Tienen distinto animal_id: {(~dups.duplicated(subset=["animal_id"])).sum():,}')

print('\n=== Ejemplo: mismo perfil, ¿mismo o distinto animal? ===')
ejemplo = dups[cols_check].head(20)
print(ejemplo.to_string())

KeyError: Index(['age_days'], dtype='str')

In [ ]:
#Posible Error de Registro
dups_diff_id = dups[~dups.duplicated(subset=['animal_id'], keep=False)]
dups_diff_id['animal_id_num'] = dups_diff_id['animal_id'].str.extract(r'(\d+)').astype(int)
dups_diff_id = dups_diff_id.sort_values(['breed_intake', 'datetime_intake', 'animal_id_num'])

# Ver cuántos tienen IDs consecutivos con mismo datetime_intake
dups_diff_id['id_consecutivo'] = (
    dups_diff_id.groupby(['breed_intake', 'datetime_intake'])['animal_id_num']
    .diff().abs() == 1
)
print(f'Pares con ID consecutivo y mismo intake: {dups_diff_id["id_consecutivo"].sum():,}')
print(dups_diff_id[dups_diff_id['id_consecutivo']][['animal_id','datetime_intake','breed_intake','outcome_type']].head(20).to_string())

Pares con ID consecutivo y mismo intake: 14,846
       animal_id     datetime_intake                         breed_intake     outcome_type
89954    A732795 2016-08-13 11:56:00                 Airedale Terrier Mix         Adoption
49774    A795382 2019-05-20 11:08:00                 Airedale Terrier Mix         Adoption
41909    A774772 2018-06-20 08:21:00                    Alaskan Husky Mix         Adoption
72252    A779262 2018-08-28 18:55:00                    Alaskan Husky Mix  Return To Owner
13664    A791100 2019-03-21 12:41:00        Alaskan Husky/German Shepherd         Transfer
16304    A791101 2019-03-21 12:41:00        Alaskan Husky/German Shepherd         Transfer
36858    A791102 2019-03-21 12:41:00        Alaskan Husky/German Shepherd         Transfer
105126   A791103 2019-03-21 12:41:00        Alaskan Husky/German Shepherd         Transfer
97260    A791104 2019-03-21 12:41:00        Alaskan Husky/German Shepherd         Transfer
74872    A791105 2019-03-21 12:41:00      

In [ ]:
# ── Eliminar solo errores de merge (outcome imposible) ────────────────────────
before = len(df)

df['datetime_intake']  = pd.to_datetime(df['datetime_intake'],  errors='coerce')
df['datetime_outcome'] = pd.to_datetime(df['datetime_outcome'], errors='coerce')

df = df[df['datetime_outcome'] >= df['datetime_intake']]

after = len(df)
print(f'Filas eliminadas (outcome < intake): {before - after:,}')
print(f'Filas restantes: {after:,}')
print(f'Duplicados restantes en EDA: estos son animales distintos con mismo perfil — son válidos')

Filas eliminadas (outcome < intake): 13,801
Filas restantes: 109,859
Duplicados restantes en EDA: estos son animales distintos con mismo perfil — son válidos


---
## 5. Diagnóstico de nulos

In [ ]:
null_summary = pd.DataFrame({
    'nulos':      df.isnull().sum(),
    'porcentaje': (df.isnull().mean() * 100).round(2)
}).sort_values('porcentaje', ascending=False)

null_summary = null_summary[null_summary['nulos'] > 0]
print(f'Columnas con nulos: {len(null_summary)}')
display(null_summary.style.bar(subset=['porcentaje'], color='#d65f5f'))

Columnas con nulos: 7


,nulos,porcentaje
outcome_subtype,66821,54.040000
name_outcome,38952,31.500000
name_intake,38952,31.500000
outcome_type,20,0.020000
age_upon_outcome,5,0.000000
sex_upon_outcome,1,0.000000
sex_upon_intake,1,0.000000


---
## 6. Tratamiento de nulos

In [ ]:
# 6.1 Columnas con >50% nulos → eliminar
HIGH_NULL_THRESHOLD = 50
cols_to_drop = null_summary[null_summary['porcentaje'] > HIGH_NULL_THRESHOLD].index.tolist()
print(f'Columnas con >{HIGH_NULL_THRESHOLD}% nulos (eliminadas): {cols_to_drop}')
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 6.2 name → 'Unknown'
for col in [c for c in df.columns if 'name' in c]:
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: nulos → "Unknown"')

# 6.3 outcome_subtype → 'None'
if 'outcome_subtype' in df.columns:
    df['outcome_subtype'] = df['outcome_subtype'].fillna('None')
    print('outcome_subtype: nulos → "None"')

# 6.4 sex_upon_intake y sex_upon_outcome → moda
for col in ['sex_upon_intake', 'sex_upon_outcome']:
    if col in df.columns and df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: nulos → "{mode_val}" (moda)')

# 6.5 Numéricas → mediana
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'{col}: nulos → mediana ({med:.1f})')

# 6.6 Categóricas restantes → 'Unknown'
for col in df.columns[df.isnull().any()]:
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: nulos → "Unknown"')

print(f'\n✅ Nulos restantes: {df.isnull().sum().sum()}')

Columnas con >50% nulos (eliminadas): ['outcome_subtype']
name_outcome: nulos → "Unknown"
name_intake: nulos → "Unknown"
sex_upon_intake: nulos → "Intact Male" (moda)
sex_upon_outcome: nulos → "Neutered Male" (moda)
outcome_type: nulos → "Unknown"
age_upon_outcome: nulos → "Unknown"

✅ Nulos restantes: 0


---
## 7. Parseo de fechas

In [ ]:
# Parsear datetime_intake y datetime_outcome
DATE_COLS = {
    'datetime_outcome': ['datetime_outcome', 'outcome_datetime'],
    'datetime_intake':  ['datetime_intake',  'intake_datetime'],
}

for alias, candidates in DATE_COLS.items():
    for c in candidates:
        if c in df.columns:
            df[alias] = pd.to_datetime(df[c], errors='coerce')
            print(f'✅ {alias} → parseado desde "{c}"')
            break
    else:
        print(f'⚠️  No se encontró columna para {alias}')

✅ datetime_outcome → parseado desde "datetime_outcome"
✅ datetime_intake → parseado desde "datetime_intake"


---
## 8. Feature engineering — todo desde INTAKE
> ✅ Corrección aplicada: todas las features temporales y de sexo/edad  
> se calculan desde las columnas de **ingreso**, no de outcome.

In [ ]:
# ── 8.1 Features temporales desde datetime_INTAKE ─────────────────────────────
# ✅ CORRECCIÓN 4 y 5: season e is_weekend desde intake (no outcome)
if 'datetime_intake' in df.columns:
    df['intake_year']  = df['datetime_intake'].dt.year
    df['intake_month'] = df['datetime_intake'].dt.month
    df['intake_dow']   = df['datetime_intake'].dt.dayofweek
    df['intake_hour']  = df['datetime_intake'].dt.hour

    # ✅ CORRECCIÓN 5: is_weekend desde intake
    df['is_weekend'] = (df['intake_dow'] >= 5).astype(int)
    print('✅ Features temporales de ingreso creadas: intake_year, intake_month, intake_dow, intake_hour, is_weekend')
else:
    print('⚠️  datetime_intake no encontrado')

# Features temporales de outcome (para referencia / análisis, NO para el modelo)
if 'datetime_outcome' in df.columns:
    df['outcome_year']  = df['datetime_outcome'].dt.year
    df['outcome_month'] = df['datetime_outcome'].dt.month
    df['outcome_dow']   = df['datetime_outcome'].dt.dayofweek
    df['outcome_hour']  = df['datetime_outcome'].dt.hour
    print('✅ Features temporales de outcome creadas (solo referencia)')

✅ Features temporales de ingreso creadas: intake_year, intake_month, intake_dow, intake_hour, is_weekend
✅ Features temporales de outcome creadas (solo referencia)


In [ ]:
# ── 8.2 Tiempo en el refugio (referencia, NO feature del modelo) ──────────────
if 'datetime_outcome' in df.columns and 'datetime_intake' in df.columns:
    df['days_in_shelter'] = (df['datetime_outcome'] - df['datetime_intake']).dt.days
    invalid = (df['days_in_shelter'] < 0).sum()
    df.loc[df['days_in_shelter'] < 0, 'days_in_shelter'] = np.nan
    df['days_in_shelter'] = df['days_in_shelter'].fillna(df['days_in_shelter'].median())
    print(f'✅ days_in_shelter creada  |  Valores negativos corregidos: {invalid}')
    print(df['days_in_shelter'].describe().round(1))

✅ days_in_shelter creada  |  Valores negativos corregidos: 13801
count    123660.0
mean         16.6
std          41.3
min           0.0
25%           2.0
50%           5.0
75%          12.0
max        1521.0
Name: days_in_shelter, dtype: float64


In [ ]:
# ── 8.3 Season desde mes de INGRESO ──────────────────────────────────────────
# ✅ CORRECCIÓN 4: season desde intake_month (no outcome_month)
def month_to_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]:  return 'Spring'
    elif month in [6, 7, 8]:  return 'Summer'
    else:                     return 'Fall'

if 'intake_month' in df.columns:
    df['season'] = df['intake_month'].apply(month_to_season)
    print('✅ season creada desde intake_month')
    print(df['season'].value_counts())
else:
    print('⚠️  intake_month no disponible')

# One-Hot Encoding de season
season_dummies = pd.get_dummies(df['season'], prefix='season', dtype=int)
df = pd.concat([df, season_dummies], axis=1)
print('\n✅ OHE de season añadido:', season_dummies.columns.tolist())

✅ season creada desde intake_month
season
Summer    33553
Fall      32572
Spring    30894
Winter    26641
Name: count, dtype: int64

✅ OHE de season añadido: ['season_Fall', 'season_Spring', 'season_Summer', 'season_Winter']


In [ ]:
# ── 8.4 Edad en días desde age_upon_INTAKE ────────────────────────────────────
# ✅ CORRECCIÓN 2: age_days desde age_upon_intake (no age_upon_outcome)
def age_to_days(age_str):
    if pd.isnull(age_str):
        return np.nan
    age_str = str(age_str).strip().lower()
    multipliers = {
        'year': 365,  'years': 365,
        'month': 30,  'months': 30,
        'week': 7,    'weeks': 7,
        'day': 1,     'days': 1
    }
    parts = age_str.split()
    if len(parts) == 2:
        try:
            number, unit = parts
            return float(number) * multipliers.get(unit, np.nan)
        except:
            return np.nan
    return np.nan

# ✅ Prioriza age_upon_intake
age_col = None
for c in ['age_upon_intake', 'age_upon_intake_intake', 'age_upon_outcome']:
    if c in df.columns:
        age_col = c
        break

if age_col:
    df['age_days'] = df[age_col].apply(age_to_days)
    df['age_days'] = df['age_days'].fillna(df['age_days'].median())
    print(f'✅ age_days creada desde "{age_col}"')
    print(df['age_days'].describe().round(1))
else:
    print('⚠️  No se encontró columna de edad de ingreso')

✅ age_days creada desde "age_upon_intake"
count    123660.0
mean        777.9
std        1064.9
min        -365.0
25%          60.0
50%         365.0
75%        1095.0
max        9125.0
Name: age_days, dtype: float64


In [ ]:
# ── 8.5 is_mix desde breed_INTAKE ────────────────────────────────────────────
# ✅ CORRECCIÓN 3: prioriza breed_intake (no breed_outcome)
breed_col = None
for c in ['breed_intake', 'breed_outcome', 'breed']:
    if c in df.columns:
        breed_col = c
        break

if breed_col:
    df['is_mix'] = df[breed_col].str.lower().str.contains('mix', na=False).astype(int)
    print(f'✅ is_mix creada desde "{breed_col}"')
    print(df['is_mix'].value_counts())

✅ is_mix creada desde "breed_intake"
is_mix
1    90643
0    33017
Name: count, dtype: int64


In [ ]:
# ── 8.6 sex y repro_status desde sex_upon_INTAKE ─────────────────────────────
# ✅ CORRECCIÓN 1: parsea desde sex_upon_intake (no sex_upon_outcome)
def parse_sex(val):
    """
    Parsea valores como: 'Intact Male', 'Spayed Female',
    'Neutered Male', 'Intact Female', 'Unknown'
    """
    val = str(val).strip().lower()
    # Sexo
    if 'female' in val:
        sex = 'Female'
    elif 'male' in val:
        sex = 'Male'
    else:
        sex = 'Unknown'
    # Estado reproductivo
    if 'neutered' in val or 'spayed' in val:
        repro = 'Fixed'
    elif 'intact' in val:
        repro = 'Intact'
    else:
        repro = 'Unknown'
    return sex, repro

# ✅ Prioriza sex_upon_intake
sex_col = None
for c in ['sex_upon_intake', 'sex_upon_intake_intake', 'sex_upon_outcome']:
    if c in df.columns:
        sex_col = c
        break

if sex_col:
    df[['sex', 'repro_status']] = df[sex_col].apply(
        lambda x: pd.Series(parse_sex(x))
    )
    print(f'✅ sex y repro_status creadas desde "{sex_col}"')
    print('\nsex:')
    print(df['sex'].value_counts())
    print('\nrepro_status:')
    print(df['repro_status'].value_counts())
else:
    print('⚠️  No se encontró columna de sexo de ingreso')

✅ sex y repro_status creadas desde "sex_upon_intake"

sex:
sex
Male       59188
Female     54255
Unknown    10217
Name: count, dtype: int64

repro_status:
repro_status
Intact     72221
Fixed      41222
Unknown    10217
Name: count, dtype: int64


---
## 9. Limpieza del target y encoding

In [ ]:
TARGET = 'outcome_type'

print('Valores únicos en outcome_type:')
print(df[TARGET].value_counts())

# Eliminar filas donde el target sea nulo
before = len(df)
df = df.dropna(subset=[TARGET])
print(f'\nFilas eliminadas por target nulo: {before - len(df)}')

# Estandarizar a Title Case
df[TARGET] = df[TARGET].str.strip().str.title()

# Label Encoding
le = LabelEncoder()
df['outcome_type_encoded'] = le.fit_transform(df[TARGET])

print('\nMapeo de clases:')
for k, v in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {v} → {k}')

Valores únicos en outcome_type:
outcome_type
Adoption           54805
Transfer           36505
Return to Owner    21485
Euthanasia          8346
Died                1146
Rto-Adopt            697
Disposal             567
Missing               68
Relocate              21
Unknown               20
Name: count, dtype: int64

Filas eliminadas por target nulo: 0

Mapeo de clases:
  0 → Adoption
  1 → Died
  2 → Disposal
  3 → Euthanasia
  4 → Missing
  5 → Relocate
  6 → Return To Owner
  7 → Rto-Adopt
  8 → Transfer
  9 → Unknown


---
## 10. Resumen del dataset final

In [ ]:
print('=' * 60)
print('       📊 RESUMEN DEL DATASET PREPROCESADO')
print('=' * 60)
print(f'Shape final    : {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Nulos restantes: {df.isnull().sum().sum()}')
print(f'Duplicados     : {df.duplicated().sum()}')

print('\n── Features creadas ─────────────────────────────────────')
new_features = [
    ('intake_month',  'Mes de ingreso (fuente de season)'),
    ('is_weekend',    '✅ Desde datetime_intake'),
    ('days_in_shelter','Solo referencia — NO usar en modelo'),
    ('season',        '✅ Desde intake_month'),
    ('season_Fall',   'OHE season'),
    ('season_Spring', 'OHE season'),
    ('season_Summer', 'OHE season'),
    ('season_Winter', 'OHE season'),
    ('age_days',      '✅ Desde age_upon_intake'),
    ('is_mix',        '✅ Desde breed_intake'),
    ('sex',           '✅ Desde sex_upon_intake'),
    ('repro_status',  '✅ Desde sex_upon_intake'),
    ('outcome_type_encoded', 'Label encoding del target'),
]
for feat, desc in new_features:
    exists = '✅' if feat in df.columns else '❌'
    print(f'  {exists} {feat:<25} {desc}')

print('\n── Distribución del target ──────────────────────────────')
print(df[TARGET].value_counts())

print('\n── Verificación: sex desde intake ───────────────────────')
if 'sex' in df.columns:
    print(df['sex'].value_counts())
    assert 'Female' in df['sex'].values, '⚠️ Female no encontrada en sex'
    print('✅ Female presente — parsing correcto')

       📊 RESUMEN DEL DATASET PREPROCESADO
Shape final    : 123,660 filas × 42 columnas
Nulos restantes: 0
Duplicados     : 0

── Features creadas ─────────────────────────────────────
  ✅ intake_month              Mes de ingreso (fuente de season)
  ✅ is_weekend                ✅ Desde datetime_intake
  ✅ days_in_shelter           Solo referencia — NO usar en modelo
  ✅ season                    ✅ Desde intake_month
  ✅ season_Fall               OHE season
  ✅ season_Spring             OHE season
  ✅ season_Summer             OHE season
  ✅ season_Winter             OHE season
  ✅ age_days                  ✅ Desde age_upon_intake
  ✅ is_mix                    ✅ Desde breed_intake
  ✅ sex                       ✅ Desde sex_upon_intake
  ✅ repro_status              ✅ Desde sex_upon_intake
  ✅ outcome_type_encoded      Label encoding del target

── Distribución del target ──────────────────────────────
outcome_type
Adoption           54805
Transfer           36505
Return To Owner    21485
E

---
## 11. Exportar dataset limpio

In [ ]:
OUTPUT_PATH = '../data/processed/austin_animal_preprocessed.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Dataset guardado en: {OUTPUT_PATH}')
print(f'   Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'\n── Columnas exportadas ──')
for c in df.columns:
    print(f'   {c}')

✅ Dataset guardado en: ../data/processed/austin_animal_preprocessed.csv
   Shape: 123,660 filas × 42 columnas

── Columnas exportadas ──
   animal_id
   name_outcome
   datetime_outcome
   monthyear_outcome
   date_of_birth
   outcome_type
   animal_type_outcome
   sex_upon_outcome
   age_upon_outcome
   breed_outcome
   color_outcome
   name_intake
   datetime_intake
   monthyear_intake
   found_location
   intake_type
   intake_condition
   animal_type_intake
   sex_upon_intake
   age_upon_intake
   breed_intake
   color_intake
   intake_year
   intake_month
   intake_dow
   intake_hour
   is_weekend
   outcome_year
   outcome_month
   outcome_dow
   outcome_hour
   days_in_shelter
   season
   season_Fall
   season_Spring
   season_Summer
   season_Winter
   age_days
   is_mix
   sex
   repro_status
   outcome_type_encoded
